# Optiver Realized Volatility Forecasting

In this project, I model short-horizon realized volatility from limit-order-book and trade data. My focus is not only the final score, but also a research workflow that would stand up in a quantitative interview: clear feature rationale, conservative validation, benchmark comparison, and error analysis.

Realized volatility is a core input for market making, risk limits, execution quality, and derivatives pricing. The dataset is also a good test of whether a modeler understands high-frequency market structure rather than only applying a generic tabular model.

What I built:
- A feature pipeline that converts raw book/trade parquet partitions into WAP, spread, depth, imbalance, return-moment, temporal-window, and cross-sectional context features.
- A leakage-aware validation setup using `GroupKFold` by `time_id`, so the same market event never appears in both train and validation.
- Fold-safe stock-level context features and batch-level time context features, matching the information boundary available at prediction time.
- A LightGBM main model trained on log-volatility and evaluated with RMSPE, the competition metric.
- Optional Optuna tuning and model comparison against CatBoost and a tabular neural baseline, using the same grouped split logic.
- Feature importance, residual diagnostics, difficult-stock analysis, and a simple realized-volatility benchmark.


## 1. Setup

The notebook runs on Kaggle by default. For local execution, set `OPTIVER_DATA_DIR` to the folder containing `train.csv`, `test.csv`, `book_train.parquet`, `book_test.parquet`, `trade_train.parquet`, and `trade_test.parquet`.

Optional environment variables:
- `OPTIVER_DATA_DIR`: competition data directory.
- `OPTIVER_WORK_DIR`: cache and submission output directory.
- `REBUILD_FEATURES=1`: force rebuilding feature parquet caches.
- `DEBUG_STOCK_LIMIT=<n>`: process only the first `n` stock partitions for a quick smoke test.
- `RUN_TUNING=1`: run Optuna hyperparameter tuning.
- `OPTUNA_TRIALS=<n>`: number of tuning trials; default is 15.
- `RUN_MODEL_COMPARISON=1`: run CatBoost and neural-network baseline comparison.


In [ ]:
import gc
import glob
import os
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.compose import ColumnTransformer
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 30)
sns.set_theme(style="whitegrid", context="notebook")

DEFAULT_DATA_DIR = Path("/kaggle/input/competitions/optiver-realized-volatility-prediction")
DEFAULT_WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")


def env_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y"}


def env_int(name, default=None):
    value = os.environ.get(name)
    if value in (None, ""):
        return default
    return int(value)


@dataclass(frozen=True)
class Config:
    data_dir: Path = Path(os.environ.get("OPTIVER_DATA_DIR", DEFAULT_DATA_DIR))
    work_dir: Path = Path(os.environ.get("OPTIVER_WORK_DIR", DEFAULT_WORK_DIR))
    seed: int = 42
    n_splits: int = 5
    rebuild_features: bool = env_flag("REBUILD_FEATURES", default=False)
    debug_stock_limit: int = env_int("DEBUG_STOCK_LIMIT", default=None)
    run_tuning: bool = env_flag("RUN_TUNING", default=False)
    optuna_trials: int = env_int("OPTUNA_TRIALS", default=15)
    run_model_comparison: bool = env_flag("RUN_MODEL_COMPARISON", default=False)

    @property
    def train_csv_path(self):
        return self.data_dir / "train.csv"

    @property
    def test_csv_path(self):
        return self.data_dir / "test.csv"

    @property
    def book_train_dir(self):
        return self.data_dir / "book_train.parquet"

    @property
    def book_test_dir(self):
        return self.data_dir / "book_test.parquet"

    @property
    def trade_train_dir(self):
        return self.data_dir / "trade_train.parquet"

    @property
    def trade_test_dir(self):
        return self.data_dir / "trade_test.parquet"

    @property
    def train_base_cache(self):
        suffix = f"_debug{self.debug_stock_limit}" if self.debug_stock_limit else ""
        return self.work_dir / f"train_base_features{suffix}.parquet"

    @property
    def test_base_cache(self):
        suffix = f"_debug{self.debug_stock_limit}" if self.debug_stock_limit else ""
        return self.work_dir / f"test_base_features{suffix}.parquet"

    @property
    def submission_path(self):
        suffix = f"_debug{self.debug_stock_limit}" if self.debug_stock_limit else ""
        return self.work_dir / f"submission_lgbm_leakage_aware{suffix}.csv"


cfg = Config()
cfg.work_dir.mkdir(parents=True, exist_ok=True)

print(f"data_dir: {cfg.data_dir}")
print(f"work_dir: {cfg.work_dir}")
print(f"rebuild_features: {cfg.rebuild_features}")
print(f"debug_stock_limit: {cfg.debug_stock_limit}")
print(f"run_tuning: {cfg.run_tuning}")
print(f"optuna_trials: {cfg.optuna_trials}")
print(f"run_model_comparison: {cfg.run_model_comparison}")


## 2. Data Contract and Metric

Each row is a `(stock_id, time_id)` prediction problem. Book and trade partitions provide intra-bucket market microstructure observations; the target is realized volatility. I use RMSPE because it matches the competition objective and makes the evaluation relative to each row's volatility level, which is more appropriate than plain RMSE for this target.


In [ ]:
def assert_data_available(config):
    required_paths = [
        config.train_csv_path,
        config.test_csv_path,
        config.book_train_dir,
        config.book_test_dir,
        config.trade_train_dir,
        config.trade_test_dir,
    ]
    missing = [str(path) for path in required_paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Competition data was not found. Set OPTIVER_DATA_DIR to the Optiver data folder. "
            f"Missing paths: {missing}"
        )


def list_stock_partitions(path, limit=None):
    files = sorted(glob.glob(str(path / "stock_id=*")))
    if limit:
        files = files[:limit]
    if not files:
        raise FileNotFoundError(f"No stock partitions found under {path}")
    return files


def rmspe(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = np.maximum(np.abs(y_true), 1e-12)
    return np.sqrt(np.mean(np.square((y_true - y_pred) / denominator)))


assert_data_available(cfg)

train_meta = pd.read_csv(cfg.train_csv_path)
test_meta = pd.read_csv(cfg.test_csv_path)

print("train.csv shape:", train_meta.shape)
print("test.csv shape:", test_meta.shape)
print("train stock count:", train_meta["stock_id"].nunique())
print("train time_id count:", train_meta["time_id"].nunique())
display(train_meta.head())
display(test_meta.head())


## 3. Microstructure Feature Engineering

The feature set is based on market microstructure signals that should plausibly explain near-term volatility:

- **Price formation:** weighted average prices, spreads, WAP balance, and short-horizon log returns.
- **Liquidity and pressure:** depth, size imbalance, order-book pressure, trade size, and order count.
- **Volatility shape:** realized volatility, absolute variation, skew, kurtosis, and six 100-second realized-volatility bins.
- **Recency:** features over the full 600-second bucket and tail windows, because later observations should be more informative for the next horizon.


In [ ]:
TAIL_WINDOWS = {
    "full": 0,
    "tail_500": 100,
    "tail_400": 200,
    "tail_300": 300,
    "tail_200": 400,
    "tail_100": 500,
}


def extract_stock_id(file_path):
    match = re.search(r"stock_id=(\d+)", str(file_path))
    if match is None:
        raise ValueError(f"Cannot parse stock_id from path: {file_path}")
    return int(match.group(1))


def safe_divide(numerator, denominator, eps=1e-12):
    denominator = denominator.where(np.abs(denominator) > eps, np.nan)
    return numerator / denominator


def log_return(series):
    return np.log(series).diff()


def realized_volatility(series):
    x = series.dropna().to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan
    return np.sqrt(np.sum(x ** 2))


def realized_absvar(series):
    x = series.dropna().to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan
    return np.sum(np.abs(x))


def realized_skew(series):
    x = series.dropna().to_numpy(dtype=float)
    if len(x) < 3:
        return 0.0
    rv = np.sqrt(np.sum(x ** 2))
    if rv == 0:
        return 0.0
    return np.sum(x ** 3) / (rv ** 3)


def realized_kurtosis(series):
    x = series.dropna().to_numpy(dtype=float)
    if len(x) < 4:
        return 0.0
    rv = np.sqrt(np.sum(x ** 2))
    if rv == 0:
        return 0.0
    return np.sum(x ** 4) / (rv ** 4)


def calc_row_slope(values):
    values = np.asarray(values, dtype=float)
    valid_idx = np.where(~np.isnan(values))[0]
    if len(valid_idx) < 2:
        return 0.0
    x = valid_idx.astype(float)
    y = values[valid_idx]
    if np.allclose(np.std(y), 0):
        return 0.0
    return float(np.polyfit(x, y, 1)[0])


def calc_wap1(df):
    denom = df["bid_size1"] + df["ask_size1"]
    return safe_divide(
        df["bid_price1"] * df["ask_size1"] + df["ask_price1"] * df["bid_size1"],
        denom,
    )


def calc_wap2(df):
    denom = df["bid_size2"] + df["ask_size2"]
    return safe_divide(
        df["bid_price2"] * df["ask_size2"] + df["ask_price2"] * df["bid_size2"],
        denom,
    )


In [ ]:
def make_book_window_features(wdf, suffix):
    if wdf.empty:
        return pd.DataFrame(columns=["time_id"])

    grp = wdf.groupby("time_id", sort=False)

    ret1 = grp["log_return1"].agg(
        rv1=realized_volatility,
        absvar1=realized_absvar,
        rskew1=realized_skew,
        rkurt1=realized_kurtosis,
    )
    ret2 = grp["log_return2"].agg(
        rv2=realized_volatility,
        absvar2=realized_absvar,
        rskew2=realized_skew,
        rkurt2=realized_kurtosis,
    )

    other = grp.agg(
        spread1_mean=("price_spread1", "mean"),
        spread1_std=("price_spread1", "std"),
        spread1_max=("price_spread1", "max"),
        spread2_mean=("price_spread2", "mean"),
        spread2_std=("price_spread2", "std"),
        spread2_max=("price_spread2", "max"),
        bid_ask_spread_mean=("bid_ask_spread", "mean"),
        bid_ask_spread_std=("bid_ask_spread", "std"),
        wap_balance_mean=("wap_balance", "mean"),
        wap_balance_std=("wap_balance", "std"),
        total_volume_mean=("total_volume", "mean"),
        total_volume_std=("total_volume", "std"),
        total_volume_sum=("total_volume", "sum"),
        imbalance_mean=("volume_imbalance", "mean"),
        imbalance_std=("volume_imbalance", "std"),
        imbalance_abs_mean=("volume_imbalance_abs", "mean"),
        depth1_mean=("depth1", "mean"),
        depth2_mean=("depth2", "mean"),
        depth_ratio_mean=("depth_ratio", "mean"),
        book_pressure1_mean=("book_pressure1", "mean"),
        book_pressure1_std=("book_pressure1", "std"),
        tau_mean=("tau", "mean"),
        tau_std=("tau", "std"),
        n_updates=("seconds_in_bucket", "count"),
        wap1_mean=("wap1", "mean"),
        wap1_std=("wap1", "std"),
        wap1_last=("wap1", "last"),
        wap2_mean=("wap2", "mean"),
        wap2_std=("wap2", "std"),
        wap2_last=("wap2", "last"),
    )

    feat = ret1.join(ret2).join(other).reset_index()
    feat = feat.rename(columns={c: f"{c}_{suffix}" for c in feat.columns if c != "time_id"})
    return feat


def make_book_bin_features(df):
    tmp = df[["time_id", "seconds_in_bucket", "log_return1"]].dropna().copy()
    if tmp.empty:
        return pd.DataFrame(columns=["time_id"])

    tmp["bin_id"] = (tmp["seconds_in_bucket"] // 100).clip(0, 5).astype(int)
    bin_rv = (
        tmp.groupby(["time_id", "bin_id"])["log_return1"]
        .agg(realized_volatility)
        .unstack()
        .reindex(columns=list(range(6)))
    )
    bin_rv.columns = [f"rv1_bin_{i}" for i in bin_rv.columns]
    out = bin_rv.reset_index()

    bin_cols = [c for c in out.columns if c.startswith("rv1_bin_")]
    out["rv1_bin_mean"] = out[bin_cols].mean(axis=1)
    out["rv1_bin_std"] = out[bin_cols].std(axis=1)
    out["rv1_bin_slope"] = out[bin_cols].apply(calc_row_slope, axis=1)
    return out


def book_preprocessor(file_path):
    stock_id = extract_stock_id(file_path)
    df = pd.read_parquet(file_path)
    df = df.sort_values(["time_id", "seconds_in_bucket"]).reset_index(drop=True)

    df["wap1"] = calc_wap1(df)
    df["wap2"] = calc_wap2(df)
    df["log_return1"] = df.groupby("time_id")["wap1"].transform(log_return)
    df["log_return2"] = df.groupby("time_id")["wap2"].transform(log_return)

    mid1 = (df["ask_price1"] + df["bid_price1"]) / 2.0
    mid2 = (df["ask_price2"] + df["bid_price2"]) / 2.0
    df["price_spread1"] = safe_divide(df["ask_price1"] - df["bid_price1"], mid1)
    df["price_spread2"] = safe_divide(df["ask_price2"] - df["bid_price2"], mid2)
    df["bid_ask_spread"] = (
        (df["ask_price1"] - df["bid_price1"]) + (df["ask_price2"] - df["bid_price2"])
    ) / 2.0
    df["wap_balance"] = np.abs(df["wap1"] - df["wap2"])
    df["depth1"] = df["ask_size1"] + df["bid_size1"]
    df["depth2"] = df["ask_size2"] + df["bid_size2"]
    df["depth_ratio"] = safe_divide(df["depth1"], df["depth2"])
    df["total_volume"] = df["depth1"] + df["depth2"]
    df["volume_imbalance"] = safe_divide(
        df["bid_size1"] + df["bid_size2"] - df["ask_size1"] - df["ask_size2"],
        df["total_volume"],
    )
    df["volume_imbalance_abs"] = np.abs(df["volume_imbalance"])
    df["book_pressure1"] = safe_divide(
        df["bid_size1"] - df["ask_size1"],
        df["bid_size1"] + df["ask_size1"],
    )
    df["tau"] = df.groupby("time_id")["seconds_in_bucket"].diff().fillna(0)

    feat = None
    for suffix, lower_bound in TAIL_WINDOWS.items():
        cur = make_book_window_features(df[df["seconds_in_bucket"] >= lower_bound], suffix)
        feat = cur if feat is None else feat.merge(cur, on="time_id", how="outer")

    feat = feat.merge(make_book_bin_features(df), on="time_id", how="left")
    feat["stock_id"] = stock_id
    cols = ["stock_id", "time_id"] + [c for c in feat.columns if c not in ["stock_id", "time_id"]]
    return feat[cols]


In [ ]:
def make_trade_window_features(wdf, suffix):
    if wdf.empty:
        return pd.DataFrame(columns=["time_id"])

    grp = wdf.groupby("time_id", sort=False)
    ret = grp["log_return"].agg(
        trade_rv=realized_volatility,
        trade_absvar=realized_absvar,
        trade_rskew=realized_skew,
        trade_rkurt=realized_kurtosis,
    )
    other = grp.agg(
        trade_size_mean=("size", "mean"),
        trade_size_std=("size", "std"),
        trade_size_sum=("size", "sum"),
        trade_size_max=("size", "max"),
        trade_order_count_mean=("order_count", "mean"),
        trade_order_count_std=("order_count", "std"),
        trade_order_count_sum=("order_count", "sum"),
        trade_order_count_max=("order_count", "max"),
        trade_price_mean=("price", "mean"),
        trade_price_std=("price", "std"),
        trade_n_prints=("seconds_in_bucket", "count"),
        trade_amount_sum=("amount", "sum"),
    )

    vwap = safe_divide(grp["amount"].sum(), grp["size"].sum()).rename("trade_vwap")
    feat = ret.join(other).join(vwap).reset_index()
    feat = feat.rename(columns={c: f"{c}_{suffix}" for c in feat.columns if c != "time_id"})
    return feat


def trade_preprocessor(file_path):
    stock_id = extract_stock_id(file_path)
    df = pd.read_parquet(file_path)
    df = df.sort_values(["time_id", "seconds_in_bucket"]).reset_index(drop=True)

    df["log_return"] = df.groupby("time_id")["price"].transform(log_return)
    df["amount"] = df["price"] * df["size"]

    feat = None
    for suffix, lower_bound in TAIL_WINDOWS.items():
        cur = make_trade_window_features(df[df["seconds_in_bucket"] >= lower_bound], suffix)
        feat = cur if feat is None else feat.merge(cur, on="time_id", how="outer")

    feat["stock_id"] = stock_id
    cols = ["stock_id", "time_id"] + [c for c in feat.columns if c not in ["stock_id", "time_id"]]
    return feat[cols]


## 4. Build and Cache Base Features

Base features are computed only from each row's own book/trade history. Context features are added later inside each fold so validation remains aligned with how the model would behave on unseen `time_id` batches.


In [ ]:
def run_preprocessor(file_list, preprocessor, label):
    feat_list = []
    total = len(file_list)

    for i, file_path in enumerate(file_list, start=1):
        print(f"{label}: {i}/{total}", end="\r")
        feat_list.append(preprocessor(file_path))
        if i % 10 == 0:
            gc.collect()

    print(f"{label}: done ({total}/{total})")
    if not feat_list:
        return pd.DataFrame(columns=["stock_id", "time_id"])
    return pd.concat(feat_list, ignore_index=True)


def merge_feature_blocks(base_df, *blocks):
    out = base_df.copy()
    for block in blocks:
        if block.empty:
            continue
        out = out.merge(block, on=["stock_id", "time_id"], how="left")
    return out


def cache_is_usable(train_df, test_df):
    only_train = sorted(set(train_df.columns) - set(test_df.columns) - {"target"})
    only_test = sorted(set(test_df.columns) - set(train_df.columns) - {"row_id"})
    return len(only_train) == 0 and len(only_test) == 0


def build_base_features(config):
    train = pd.read_csv(config.train_csv_path)
    test = pd.read_csv(config.test_csv_path)

    train_book_files = list_stock_partitions(config.book_train_dir, config.debug_stock_limit)
    train_trade_files = list_stock_partitions(config.trade_train_dir, config.debug_stock_limit)
    test_book_files = list_stock_partitions(config.book_test_dir, config.debug_stock_limit)
    test_trade_files = list_stock_partitions(config.trade_test_dir, config.debug_stock_limit)

    print("train book files:", len(train_book_files))
    print("train trade files:", len(train_trade_files))
    print("test book files:", len(test_book_files))
    print("test trade files:", len(test_trade_files))

    book_train = run_preprocessor(train_book_files, book_preprocessor, "book train")
    trade_train = run_preprocessor(train_trade_files, trade_preprocessor, "trade train")
    book_test = run_preprocessor(test_book_files, book_preprocessor, "book test")
    trade_test = run_preprocessor(test_trade_files, trade_preprocessor, "trade test")

    train_base = merge_feature_blocks(train, book_train, trade_train)
    test_base = merge_feature_blocks(test, book_test, trade_test)
    return train_base, test_base


use_cache = (
    not cfg.rebuild_features
    and cfg.train_base_cache.exists()
    and cfg.test_base_cache.exists()
)

if use_cache:
    print("Loading cached base features...")
    train_base = pd.read_parquet(cfg.train_base_cache)
    test_base = pd.read_parquet(cfg.test_base_cache)
    if not cache_is_usable(train_base, test_base):
        print("Cached feature schemas are inconsistent; rebuilding.")
        use_cache = False

if not use_cache:
    train_base, test_base = build_base_features(cfg)
    train_base.to_parquet(cfg.train_base_cache, index=False)
    test_base.to_parquet(cfg.test_base_cache, index=False)
    print("Base feature cache saved.")

print("train_base shape:", train_base.shape)
print("test_base shape:", test_base.shape)
print("base feature count:", len(set(train_base.columns) & set(test_base.columns)) - 3)


## 5. Exploratory Checks

These checks make the project readable to reviewers: they show target scale, stock heterogeneity, missingness, and how much signal a simple realized-volatility proxy already carries.


In [ ]:
target_summary = train_base["target"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
display(target_summary.to_frame("target"))

benchmark_cols = [c for c in ["rv1_full", "rv2_full", "trade_rv_full"] if c in train_base.columns]
simple_benchmark = train_base[benchmark_cols].mean(axis=1)
simple_benchmark = simple_benchmark.fillna(simple_benchmark.median()).to_numpy()
print("benchmark columns:", benchmark_cols)
print("simple realized-volatility benchmark RMSPE:", round(float(rmspe(train_base["target"], simple_benchmark)), 5))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

sns.histplot(train_base["target"], bins=80, ax=axes[0])
axes[0].set_title("Target distribution")
axes[0].set_xlabel("realized volatility")

sns.histplot(np.log(train_base["target"]), bins=80, ax=axes[1])
axes[1].set_title("Log target distribution")
axes[1].set_xlabel("log(realized volatility)")

stock_target = train_base.groupby("stock_id")["target"].mean().sort_values()
sns.lineplot(x=np.arange(len(stock_target)), y=stock_target.values, ax=axes[2])
axes[2].set_title("Average target by stock")
axes[2].set_xlabel("stocks sorted by mean target")
axes[2].set_ylabel("mean target")

plt.tight_layout()
plt.show()


## 6. Leakage-Aware Context Features

Cross-sectional context is useful in this competition because each `time_id` contains many stocks observed under the same market conditions.

Validation design:
- `GroupKFold` holds out entire `time_id` groups.
- Time-level features are computed within the prediction batch. This mirrors test-time availability, where all rows in a submitted batch are known.
- Stock-level historical averages are fit on the training fold only, then applied to validation/test. This avoids allowing validation rows to influence their own stock priors.


In [ ]:
FEATURE_EXCLUDE_COLS = {"row_id", "target", "time_id"}
MISSING_FLAG_EXCLUDE_COLS = {"row_id", "target", "stock_id", "time_id"}

CONTEXT_KEYWORDS = [
    "rv1_", "rv2_", "absvar", "rskew", "rkurt", "spread", "wap_balance",
    "total_volume", "imbalance", "depth", "pressure", "tau", "n_updates",
    "trade_rv", "trade_absvar", "trade_rskew", "trade_rkurt", "trade_size",
    "trade_order_count", "trade_price", "trade_amount", "trade_vwap",
    "trade_n_prints", "rv1_bin_",
]

DIFF_SEED_CANDIDATES = [
    "rv1_full", "rv2_full", "absvar1_full", "absvar2_full",
    "spread1_mean_full", "spread2_mean_full", "bid_ask_spread_mean_full",
    "wap_balance_mean_full", "total_volume_mean_full", "imbalance_mean_full",
    "depth1_mean_full", "depth2_mean_full", "book_pressure1_mean_full",
    "tau_mean_full", "trade_rv_full", "trade_absvar_full", "trade_rskew_full",
    "trade_size_mean_full", "trade_order_count_mean_full", "trade_price_mean_full",
    "trade_amount_sum_full", "trade_vwap_full", "rv1_bin_mean", "rv1_bin_slope",
]


def add_missing_flags(df):
    out = df.copy()
    feature_cols = [c for c in out.columns if c not in MISSING_FLAG_EXCLUDE_COLS]
    out["row_missing_count"] = out[feature_cols].isna().sum(axis=1).astype("int16")
    out["row_missing_ratio"] = out[feature_cols].isna().mean(axis=1).astype("float32")
    return out


def context_seed_columns(train_df, test_df):
    numeric_cols = [
        c for c in train_df.columns
        if c not in FEATURE_EXCLUDE_COLS
        and c in test_df.columns
        and pd.api.types.is_numeric_dtype(train_df[c])
    ]
    return [c for c in numeric_cols if any(key in c for key in CONTEXT_KEYWORDS)]


def flatten_agg_columns(agg_df, key):
    scope = key.replace("_id", "")
    agg_df.columns = [
        key if col[0] == key else f"{col[0]}_by_{scope}_{col[1]}"
        for col in agg_df.columns
    ]
    return agg_df


def fit_stock_context(df, seed_cols):
    stock_ids = pd.DataFrame({"stock_id": sorted(df["stock_id"].unique())})
    if not seed_cols:
        return stock_ids
    agg = df.groupby("stock_id")[seed_cols].agg(["mean", "std"]).reset_index()
    return flatten_agg_columns(agg, "stock_id")


def add_time_context(df, seed_cols):
    out = df.copy()
    if not seed_cols:
        return out
    agg = out.groupby("time_id")[seed_cols].agg(["mean", "std"]).reset_index()
    agg = flatten_agg_columns(agg, "time_id")
    return out.merge(agg, on="time_id", how="left")


def add_relative_context_features(df, seed_cols):
    out = df.copy()
    for col in [c for c in DIFF_SEED_CANDIDATES if c in seed_cols]:
        stock_mean = f"{col}_by_stock_mean"
        time_mean = f"{col}_by_time_mean"
        if stock_mean in out.columns:
            out[f"{col}_minus_stock_mean"] = out[col] - out[stock_mean]
        if time_mean in out.columns:
            out[f"{col}_minus_time_mean"] = out[col] - out[time_mean]
    return out


def apply_context_features(base_df, stock_context, seed_cols):
    out = add_time_context(base_df, seed_cols)
    out = out.merge(stock_context, on="stock_id", how="left")
    out = add_relative_context_features(out, seed_cols)
    out = add_missing_flags(out)
    out = out.loc[:, ~out.columns.duplicated()].copy()
    return out


def make_feature_columns(train_context, test_context=None):
    cols = [c for c in train_context.columns if c not in FEATURE_EXCLUDE_COLS]
    if test_context is not None:
        cols = [c for c in cols if c in test_context.columns]
    return cols


def prepare_feature_frame(df, feature_cols, fill_values=None):
    X = df.reindex(columns=feature_cols).copy()
    numeric_cols = [c for c in feature_cols if c != "stock_id"]
    X[numeric_cols] = X[numeric_cols].replace([np.inf, -np.inf], np.nan)

    if fill_values is None:
        fill_values = X[numeric_cols].median().fillna(0)

    X[numeric_cols] = X[numeric_cols].fillna(fill_values)
    if "stock_id" in X.columns:
        X["stock_id"] = X["stock_id"].astype("category")
    return X, fill_values


seed_cols = context_seed_columns(train_base, test_base)
print("context seed columns:", len(seed_cols))
print(seed_cols[:20])


## 7. Model Training

The model predicts `log(target)` and exponentiates predictions before RMSPE evaluation. This stabilizes the right-skewed target and naturally keeps predictions positive.


In [ ]:
lgbm_params = dict(
    objective="regression",
    metric="l2",
    n_estimators=5000,
    learning_rate=0.015,
    num_leaves=96,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.80,
    subsample_freq=1,
    colsample_bytree=0.80,
    reg_alpha=0.30,
    reg_lambda=1.20,
    random_state=cfg.seed,
    n_jobs=-1,
    force_col_wise=True,
    verbosity=-1,
)

groups = train_base["time_id"].to_numpy()
y = train_base["target"].to_numpy(dtype=float)
y_log = np.log(np.maximum(y, 1e-12))
gkf = GroupKFold(n_splits=cfg.n_splits)


def grouped_cv_splits(max_folds=None):
    splits = list(gkf.split(train_base, y_log, groups=groups))
    if max_folds is not None:
        splits = splits[:max_folds]
    return splits


def build_fold_matrices(train_idx, valid_idx, selected_features=None, include_test=False):
    fold_train_base = train_base.iloc[train_idx].copy()
    fold_valid_base = train_base.iloc[valid_idx].copy()

    stock_context = fit_stock_context(fold_train_base, seed_cols)
    fold_train_context = apply_context_features(fold_train_base, stock_context, seed_cols)
    fold_valid_context = apply_context_features(fold_valid_base, stock_context, seed_cols)

    fold_test_context = None
    if include_test:
        fold_test_context = apply_context_features(test_base, stock_context, seed_cols)

    if selected_features is None:
        selected_features = make_feature_columns(
            fold_train_context,
            fold_test_context if include_test else fold_valid_context,
        )

    X_train, fill_values = prepare_feature_frame(fold_train_context, selected_features)
    X_valid, _ = prepare_feature_frame(fold_valid_context, selected_features, fill_values)
    X_test = None
    if include_test:
        X_test, _ = prepare_feature_frame(fold_test_context, selected_features, fill_values)

    del fold_train_context, fold_valid_context, fold_test_context
    return X_train, X_valid, X_test, selected_features


def fit_lgbm_grouped_cv(
    params,
    selected_features=None,
    max_folds=None,
    predict_test=False,
    early_stopping_rounds=250,
    log_every=250,
    label="LightGBM",
):
    splits = grouped_cv_splits(max_folds=max_folds)
    oof = np.full(len(train_base), np.nan, dtype=float)
    test_accumulator = np.zeros(len(test_base), dtype=float) if predict_test else None
    fold_scores_local = []
    best_iters_local = []
    importance_local = []
    resolved_features = selected_features

    for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
        X_train, X_valid, X_test, resolved_features = build_fold_matrices(
            train_idx,
            valid_idx,
            selected_features=resolved_features,
            include_test=predict_test,
        )

        if fold == 1:
            print(f"{label} feature count:", len(resolved_features))

        fold_params = params.copy()
        fold_params["random_state"] = cfg.seed + fold
        model = LGBMRegressor(**fold_params)

        categorical_feature = ["stock_id"] if "stock_id" in resolved_features else "auto"
        callbacks = [early_stopping(early_stopping_rounds, verbose=False)]
        if log_every and log_every > 0:
            callbacks.append(log_evaluation(log_every))

        model.fit(
            X_train,
            y_log[train_idx],
            eval_set=[(X_valid, y_log[valid_idx])],
            eval_metric="l2",
            callbacks=callbacks,
            categorical_feature=categorical_feature,
        )

        best_iteration = model.best_iteration_ or params["n_estimators"]
        valid_pred = np.exp(model.predict(X_valid, num_iteration=best_iteration))
        valid_pred = np.clip(valid_pred, 1e-8, None)

        oof[valid_idx] = valid_pred
        if predict_test:
            test_pred = np.exp(model.predict(X_test, num_iteration=best_iteration))
            test_accumulator += np.clip(test_pred, 1e-8, None) / len(splits)

        fold_score = rmspe(y[valid_idx], valid_pred)
        fold_scores_local.append(fold_score)
        best_iters_local.append(best_iteration)

        importance_local.append(
            pd.DataFrame({
                "feature": resolved_features,
                "importance_gain": model.booster_.feature_importance(importance_type="gain"),
                "fold": fold,
            })
        )

        print(f"{label} fold {fold} RMSPE: {fold_score:.5f} | best_iter: {best_iteration}")

        del X_train, X_valid, X_test, model
        gc.collect()

    scored_mask = ~np.isnan(oof)
    return {
        "label": label,
        "oof_preds": oof,
        "test_preds": test_accumulator,
        "fold_scores": fold_scores_local,
        "oof_rmspe": rmspe(y[scored_mask], oof[scored_mask]),
        "best_iters": best_iters_local,
        "importance_dfs": importance_local,
        "feature_cols": resolved_features,
        "fold_count": len(splits),
    }


main_lgbm_result = fit_lgbm_grouped_cv(
    lgbm_params,
    predict_test=True,
    early_stopping_rounds=250,
    log_every=250,
    label="LightGBM main",
)

oof_preds = main_lgbm_result["oof_preds"]
test_preds = main_lgbm_result["test_preds"]
fold_scores = main_lgbm_result["fold_scores"]
best_iters = main_lgbm_result["best_iters"]
importance_dfs = main_lgbm_result["importance_dfs"]
feature_cols = main_lgbm_result["feature_cols"]

print()
print("fold scores:", [round(float(x), 5) for x in fold_scores])
print("mean CV RMSPE:", round(float(np.mean(fold_scores)), 5))
print("OOF RMSPE:", round(float(main_lgbm_result["oof_rmspe"]), 5))
print("mean best_iter:", int(np.mean(best_iters)))


## 8. Model Interpretation

Feature importance is aggregated across folds by gain. Reviewers should be able to connect top features back to market intuition, not just see a leaderboard score.


In [ ]:
feature_importance_df = pd.concat(importance_dfs, ignore_index=True)
feature_importance_summary = (
    feature_importance_df.groupby("feature", as_index=False)["importance_gain"]
    .mean()
    .sort_values("importance_gain", ascending=False)
)

reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)

display(feature_importance_summary.head(30))

top_n = 25
fig, ax = plt.subplots(figsize=(10, 8))
plot_df = feature_importance_summary.head(top_n).sort_values("importance_gain")
sns.barplot(data=plot_df, x="importance_gain", y="feature", ax=ax, color="#2878b5")
ax.set_title(f"Top {top_n} LightGBM features by mean gain")
ax.set_xlabel("mean gain across folds")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(reports_dir / "feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()


## 9. Optuna Tuning and Model Comparison

I keep the main model simple and reproducible, then use this section as an experimental check. Optuna tuning, CatBoost, and the neural baseline all reuse the same grouped validation split and the same fold-safe context-feature construction. To keep runtime practical, these experiments use the highest-gain features from the main LightGBM run.

This section is controlled by environment variables:
- Set `RUN_TUNING=1` to run Optuna.
- Set `RUN_MODEL_COMPARISON=1` to compare LightGBM, CatBoost, and a tabular MLP baseline.


In [ ]:
def compact_feature_set(top_n=160):
    selected = []
    if "stock_id" in feature_cols:
        selected.append("stock_id")

    for feature in feature_importance_summary["feature"]:
        if feature in feature_cols and feature not in selected:
            selected.append(feature)
        non_stock_count = sum(col != "stock_id" for col in selected)
        if non_stock_count >= top_n:
            break

    return selected


tuning_feature_cols = compact_feature_set(top_n=160)
comparison_feature_cols = compact_feature_set(top_n=120)
neural_feature_cols = compact_feature_set(top_n=80)

print("Optuna feature count:", len(tuning_feature_cols))
print("comparison feature count:", len(comparison_feature_cols))
print("neural baseline feature count:", len(neural_feature_cols))


In [ ]:
tuning_summary = pd.DataFrame()
best_lgbm_params = lgbm_params.copy()

if cfg.run_tuning:
    try:
        import optuna
    except ImportError:
        print("Optuna is not installed. Install requirements.txt or run `pip install optuna`.")
    else:
        optuna.logging.set_verbosity(optuna.logging.WARNING)

        def objective(trial):
            trial_params = lgbm_params.copy()
            trial_params.update({
                "n_estimators": 2500,
                "learning_rate": trial.suggest_float("learning_rate", 0.008, 0.04, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 48, 160),
                "min_child_samples": trial.suggest_int("min_child_samples", 20, 140),
                "subsample": trial.suggest_float("subsample", 0.65, 0.95),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 0.95),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 2.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
                "max_depth": trial.suggest_categorical("max_depth", [-1, 6, 8, 10, 12]),
            })

            result = fit_lgbm_grouped_cv(
                trial_params,
                selected_features=tuning_feature_cols,
                max_folds=min(3, cfg.n_splits),
                predict_test=False,
                early_stopping_rounds=150,
                log_every=0,
                label=f"Optuna trial {trial.number}",
            )
            return result["oof_rmspe"]

        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=cfg.seed),
        )
        study.optimize(objective, n_trials=cfg.optuna_trials, gc_after_trial=True)

        best_lgbm_params.update(study.best_params)
        best_lgbm_params["n_estimators"] = 3500

        tuning_summary = study.trials_dataframe(attrs=("number", "value", "params", "state"))
        tuning_summary = tuning_summary.sort_values("value", ascending=True).reset_index(drop=True)
        display(tuning_summary.head(10))
        print("best Optuna RMSPE:", round(float(study.best_value), 5))
        print("best Optuna params:", study.best_params)
else:
    print("Optuna tuning skipped. Set RUN_TUNING=1 to run it.")


In [ ]:
comparison_rows = [{
    "model": "LightGBM main",
    "folds": cfg.n_splits,
    "feature_count": len(feature_cols),
    "cv_rmspe": main_lgbm_result["oof_rmspe"],
    "notes": "Primary model, all engineered features",
}]


def add_comparison_row(rows, model, result, feature_count, notes):
    rows.append({
        "model": model,
        "folds": result["fold_count"],
        "feature_count": feature_count,
        "cv_rmspe": result["oof_rmspe"],
        "notes": notes,
    })


def fit_catboost_grouped_cv(params, selected_features, max_folds=None, label="CatBoost"):
    from catboost import CatBoostRegressor

    splits = grouped_cv_splits(max_folds=max_folds)
    oof = np.full(len(train_base), np.nan, dtype=float)
    fold_scores_local = []
    resolved_features = selected_features

    for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
        X_train, X_valid, _, resolved_features = build_fold_matrices(
            train_idx,
            valid_idx,
            selected_features=resolved_features,
            include_test=False,
        )

        cat_features = []
        if "stock_id" in resolved_features:
            X_train["stock_id"] = X_train["stock_id"].astype(str)
            X_valid["stock_id"] = X_valid["stock_id"].astype(str)
            cat_features = [resolved_features.index("stock_id")]

        fold_params = params.copy()
        fold_params["random_seed"] = cfg.seed + fold
        model = CatBoostRegressor(**fold_params)
        model.fit(
            X_train,
            y_log[train_idx],
            eval_set=(X_valid, y_log[valid_idx]),
            cat_features=cat_features,
            use_best_model=True,
            verbose=False,
        )

        valid_pred = np.exp(model.predict(X_valid))
        valid_pred = np.clip(valid_pred, 1e-8, None)
        oof[valid_idx] = valid_pred
        fold_score = rmspe(y[valid_idx], valid_pred)
        fold_scores_local.append(fold_score)

        print(f"{label} fold {fold} RMSPE: {fold_score:.5f}")

        del X_train, X_valid, model
        gc.collect()

    scored_mask = ~np.isnan(oof)
    return {
        "label": label,
        "oof_preds": oof,
        "fold_scores": fold_scores_local,
        "oof_rmspe": rmspe(y[scored_mask], oof[scored_mask]),
        "fold_count": len(splits),
    }


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def fit_mlp_grouped_cv(selected_features, max_folds=None, label="MLP baseline"):
    splits = grouped_cv_splits(max_folds=max_folds)
    oof = np.full(len(train_base), np.nan, dtype=float)
    fold_scores_local = []
    resolved_features = selected_features

    for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
        X_train, X_valid, _, resolved_features = build_fold_matrices(
            train_idx,
            valid_idx,
            selected_features=resolved_features,
            include_test=False,
        )

        numeric_cols = [col for col in resolved_features if col != "stock_id"]
        transformers = []
        if numeric_cols:
            transformers.append(("num", StandardScaler(), numeric_cols))
        if "stock_id" in resolved_features:
            X_train["stock_id"] = X_train["stock_id"].astype(str)
            X_valid["stock_id"] = X_valid["stock_id"].astype(str)
            transformers.append(("stock", make_one_hot_encoder(), ["stock_id"]))

        preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
        model = make_pipeline(
            preprocessor,
            MLPRegressor(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                alpha=1e-4,
                learning_rate_init=1e-3,
                early_stopping=True,
                validation_fraction=0.15,
                max_iter=120,
                random_state=cfg.seed + fold,
            ),
        )
        model.fit(X_train, y_log[train_idx])

        valid_pred = np.exp(model.predict(X_valid))
        valid_pred = np.clip(valid_pred, 1e-8, None)
        oof[valid_idx] = valid_pred
        fold_score = rmspe(y[valid_idx], valid_pred)
        fold_scores_local.append(fold_score)

        print(f"{label} fold {fold} RMSPE: {fold_score:.5f}")

        del X_train, X_valid, model
        gc.collect()

    scored_mask = ~np.isnan(oof)
    return {
        "label": label,
        "oof_preds": oof,
        "fold_scores": fold_scores_local,
        "oof_rmspe": rmspe(y[scored_mask], oof[scored_mask]),
        "fold_count": len(splits),
    }


if cfg.run_model_comparison:
    comparison_folds = min(3, cfg.n_splits)

    compact_lgbm_params = best_lgbm_params.copy()
    compact_lgbm_params["n_estimators"] = min(compact_lgbm_params.get("n_estimators", 2500), 2500)
    compact_lgbm_result = fit_lgbm_grouped_cv(
        compact_lgbm_params,
        selected_features=comparison_feature_cols,
        max_folds=comparison_folds,
        predict_test=False,
        early_stopping_rounds=150,
        log_every=0,
        label="LightGBM compact",
    )
    add_comparison_row(
        comparison_rows,
        "LightGBM compact",
        compact_lgbm_result,
        len(comparison_feature_cols),
        "Top-gain features; tuned params if Optuna was run",
    )

    try:
        catboost_params = dict(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=2500,
            learning_rate=0.03,
            depth=7,
            l2_leaf_reg=4.0,
            random_strength=0.5,
            od_type="Iter",
            od_wait=150,
            thread_count=-1,
            allow_writing_files=False,
        )
        catboost_result = fit_catboost_grouped_cv(
            catboost_params,
            selected_features=comparison_feature_cols,
            max_folds=comparison_folds,
            label="CatBoost",
        )
        add_comparison_row(
            comparison_rows,
            "CatBoost",
            catboost_result,
            len(comparison_feature_cols),
            "Ordered boosting baseline on the same grouped folds",
        )
    except ImportError:
        print("CatBoost is not installed. Install requirements.txt or run `pip install catboost`.")

    mlp_result = fit_mlp_grouped_cv(
        selected_features=neural_feature_cols,
        max_folds=comparison_folds,
        label="MLP baseline",
    )
    add_comparison_row(
        comparison_rows,
        "Tabular MLP",
        mlp_result,
        len(neural_feature_cols),
        "Scaled top-gain numeric features plus one-hot stock_id",
    )
else:
    print("Model comparison skipped. Set RUN_MODEL_COMPARISON=1 to run it.")

comparison_summary = pd.DataFrame(comparison_rows).sort_values("cv_rmspe")
display(comparison_summary)


## 10. Error Diagnostics

I include these diagnostics because a useful model is not just a single CV score. The cells below identify difficult stocks/time buckets and visualize residual behavior.


In [ ]:
analysis_df = train_base[["stock_id", "time_id", "target"]].copy()
analysis_df["oof_pred"] = oof_preds
analysis_df["ape"] = np.abs((analysis_df["target"] - analysis_df["oof_pred"]) / analysis_df["target"])
analysis_df["pct_error"] = (analysis_df["oof_pred"] - analysis_df["target"]) / analysis_df["target"]

stock_error = (
    analysis_df.groupby("stock_id")
    .agg(
        n_obs=("target", "size"),
        target_mean=("target", "mean"),
        pred_mean=("oof_pred", "mean"),
        rmspe=("ape", lambda x: np.sqrt(np.mean(np.square(x)))),
        median_ape=("ape", "median"),
    )
    .sort_values("rmspe", ascending=False)
)

time_error = (
    analysis_df.groupby("time_id")
    .agg(
        n_obs=("target", "size"),
        target_mean=("target", "mean"),
        pred_mean=("oof_pred", "mean"),
        rmspe=("ape", lambda x: np.sqrt(np.mean(np.square(x)))),
        median_ape=("ape", "median"),
    )
    .sort_values("rmspe", ascending=False)
)

print("hardest stocks")
display(stock_error.head(10))
print("easiest stocks")
display(stock_error.tail(10))
print("hardest time_ids")
display(time_error.head(10))


In [ ]:
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

sample_df = analysis_df.sample(min(len(analysis_df), 20000), random_state=cfg.seed)

sns.scatterplot(data=sample_df, x="target", y="oof_pred", alpha=0.25, s=12, ax=axes[0], edgecolor=None)
lims = [
    min(sample_df["target"].min(), sample_df["oof_pred"].min()),
    max(sample_df["target"].max(), sample_df["oof_pred"].max()),
]
axes[0].plot(lims, lims, color="black", linewidth=1)
axes[0].set_title("OOF prediction vs target")

sns.histplot(analysis_df["pct_error"].clip(-2, 2), bins=100, ax=axes[1])
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Percent error distribution")
axes[1].set_xlabel("clipped percent error")

stock_error["rmspe"].sort_values().plot(kind="bar", ax=axes[2], color="#4c956c")
axes[2].set_title("RMSPE by stock")
axes[2].set_xlabel("stock_id sorted by RMSPE")
axes[2].set_ylabel("RMSPE")
axes[2].set_xticks([])

plt.tight_layout()
fig.savefig(reports_dir / "residual_diagnostics.png", dpi=160, bbox_inches="tight")
plt.show()


## 11. Benchmark, Submission, and Experiment Card

The final output records the competition submission file and a compact experiment card. I use this table to compare future tuning runs without relying on memory or scattered notebook outputs.


In [ ]:
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)

benchmark_pred = train_base[benchmark_cols].mean(axis=1)
benchmark_pred = benchmark_pred.fillna(benchmark_pred.median()).to_numpy()

benchmark_score = rmspe(y, benchmark_pred)
oof_score = rmspe(y, oof_preds)
relative_lift = (benchmark_score - oof_score) / benchmark_score

fold_score_report = pd.DataFrame({
    "fold": np.arange(1, len(fold_scores) + 1),
    "rmspe": fold_scores,
    "best_iteration": best_iters,
})
fold_score_report.to_csv(reports_dir / "fold_scores.csv", index=False)

print("simple benchmark RMSPE:", round(float(benchmark_score), 5))
print("LightGBM OOF RMSPE:", round(float(oof_score), 5))
print("relative RMSPE reduction:", f"{relative_lift:.2%}")
print("saved:", reports_dir / "fold_scores.csv")

submission = test_base[["row_id"]].copy()
submission["target"] = np.clip(test_preds, 1e-8, None)
submission.to_csv(cfg.submission_path, index=False)

print("saved:", cfg.submission_path)
display(submission.head())


In [ ]:
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)

experiment_summary = {
    "model": "LightGBMRegressor",
    "validation": f"GroupKFold(n_splits={cfg.n_splits}) by time_id",
    "target_transform": "log(target)",
    "metric": "RMSPE",
    "feature_count": len(feature_cols),
    "context_seed_count": len(seed_cols),
    "mean_cv_rmspe": round(float(np.mean(fold_scores)), 5),
    "oof_rmspe": round(float(rmspe(y, oof_preds)), 5),
    "simple_benchmark_rmspe": round(float(benchmark_score), 5),
    "relative_rmspe_reduction_vs_benchmark": f"{relative_lift:.2%}",
    "mean_best_iter": int(np.mean(best_iters)),
    "uses_book_features": True,
    "uses_trade_features": True,
    "uses_leakage_aware_stock_context": True,
    "uses_batch_time_context": True,
    "optuna_trials_run": 0 if tuning_summary.empty else int(len(tuning_summary)),
    "model_comparison_rows": int(len(comparison_summary)),
}

top_feature_lines = "\n".join(
    f"- {row.feature}: {row.importance_gain:.2f}"
    for row in feature_importance_summary.head(10).itertuples(index=False)
)
comparison_text = comparison_summary.to_string(index=False)
summary_lines = [
    "# Model Summary",
    "",
    "## Final Results",
    "",
    f"- Baseline RMSPE: {experiment_summary['simple_benchmark_rmspe']}",
    f"- LightGBM OOF RMSPE: {experiment_summary['oof_rmspe']}",
    f"- Relative RMSPE reduction: {experiment_summary['relative_rmspe_reduction_vs_benchmark']}",
    f"- Number of engineered features: {experiment_summary['feature_count']}",
    f"- CV strategy: {experiment_summary['validation']}",
    "- Main feature groups: WAP returns, realized-volatility moments, spread/depth signals, order imbalance, tail-window features, trade-flow features, and cross-sectional context features",
    "",
    "## Top Features",
    "",
    top_feature_lines,
    "",
    "## Model Comparison",
    "",
    "```text",
    comparison_text,
    "```",
    "",
    "## Generated Files",
    "",
    "- reports/fold_scores.csv",
    "- reports/feature_importance.png",
    "- reports/residual_diagnostics.png",
    "",
    "## Limitation",
    "",
    "This is a Kaggle-style supervised forecasting project. The validation design is leakage-aware by time_id, but the model is not a production trading strategy and does not simulate transaction costs, latency, slippage, or live execution.",
    "",
]
(reports_dir / "model_summary.md").write_text("\n".join(summary_lines), encoding="utf-8")

print("saved:", reports_dir / "model_summary.md")
display(pd.Series(experiment_summary, name="experiment_card").to_frame())


## 12. Closing Notes

This version is intentionally kept as a focused research project rather than an oversized modeling framework. The strongest parts are the microstructure feature design, the grouped validation boundary, and the comparison against reasonable alternatives.

If I continued this project, I would focus on:
- Add fold-level calibration checks by volatility regime.
- Build feature families from order-flow imbalance persistence and trade intensity bursts.
- Package preprocessing into a reusable module and run the notebook from a reproducible CLI.
